# UrbanTransit IQ — Data Cleaning Notebook
**Phase 3: Data Quality Analysis & Cleaning**

This notebook imports each CSV from `raw_data/`, detects data quality issues,
cleans/corrects/flags invalid records, and saves cleaned outputs to `processed_data/`.

**Tables Cleaned:**
1. passengers.csv
2. routes.csv
3. stops.csv
4. vehicles.csv
5. route_stops.csv
6. schedules.csv
7. service_calendar.csv
8. trips.csv
9. delays.csv
10. tickets.csv
11. passenger_counts.csv
12. gps_events.csv

In [ ]:
# ─── Cell 1: Import Libraries ───────────────────────────────────────────────
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Paths
RAW_DIR        = '../raw_data/'      # Source CSVs
PROCESSED_DIR  = '../processed_data/' # Cleaned CSVs output
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Cleaning log — records every fix applied
cleaning_log = []

def log_issue(table, column, issue_type, original_count, action):
    """Append an entry to the cleaning audit log."""
    cleaning_log.append({
        'table'         : table,
        'column'        : column,
        'issue_type'    : issue_type,
        'original_count': original_count,
        'action'        : action
    })

print('Libraries loaded. RAW_DIR =', RAW_DIR)
print('PROCESSED_DIR =', PROCESSED_DIR)

---
## 1. passengers.csv

In [ ]:
# ─── Cell 2: Load passengers.csv ────────────────────────────────────────────
passengers_raw = pd.read_csv(RAW_DIR + 'passengers.csv')
print('Shape (raw):', passengers_raw.shape)
passengers_raw.head(3)

In [ ]:
# ─── Cell 3: Inspect passengers ─────────────────────────────────────────────
print('--- Null counts ---')
print(passengers_raw.isnull().sum())
print('\n--- Dtypes ---')
print(passengers_raw.dtypes)
print('\n--- Duplicates on passenger_id ---')
print(passengers_raw.duplicated(subset=['passenger_id']).sum())

In [ ]:
# ─── Cell 4: Clean passengers ───────────────────────────────────────────────
passengers = passengers_raw.copy()

# Rule P1: Drop duplicate passenger_id (keep first occurrence)
dup_count = passengers.duplicated(subset=['passenger_id']).sum()
if dup_count > 0:
    log_issue('passengers', 'passenger_id', 'duplicate_pk', dup_count, 'dropped duplicate rows')
passengers.drop_duplicates(subset=['passenger_id'], keep='first', inplace=True)

# Rule P2: Fill missing passenger_type with mode
missing_type = passengers['passenger_type'].isnull().sum()
if missing_type > 0:
    mode_val = passengers['passenger_type'].mode()[0]
    passengers['passenger_type'].fillna(mode_val, inplace=True)
    log_issue('passengers', 'passenger_type', 'missing_value', missing_type,
              f'filled with mode = {mode_val}')

# Rule P3: Parse registration_date as datetime
passengers['registration_date'] = pd.to_datetime(passengers['registration_date'], errors='coerce')
invalid_dates = passengers['registration_date'].isnull().sum()
if invalid_dates > 0:
    log_issue('passengers', 'registration_date', 'invalid_date', invalid_dates,
              'set to NaT (unparseable dates)')

print('Shape after cleaning:', passengers.shape)
passengers.head(3)

In [ ]:
# ─── Cell 5: Save passengers ─────────────────────────────────────────────────
passengers.to_csv(PROCESSED_DIR + 'passengers_clean.csv', index=False)
print('Saved: passengers_clean.csv')

---
## 2. routes.csv

In [ ]:
# ─── Cell 6: Load & inspect routes.csv ──────────────────────────────────────
routes_raw = pd.read_csv(RAW_DIR + 'routes.csv')
print('Shape (raw):', routes_raw.shape)
print('Nulls:\n', routes_raw.isnull().sum())
print('Duplicates on route_id:', routes_raw.duplicated(subset=['route_id']).sum())
routes_raw.head(3)

In [ ]:
# ─── Cell 7: Clean routes ────────────────────────────────────────────────────
routes = routes_raw.copy()

# Rule R1: Drop duplicate route_id
dup_r = routes.duplicated(subset=['route_id']).sum()
if dup_r > 0:
    log_issue('routes', 'route_id', 'duplicate_pk', dup_r, 'dropped duplicates')
routes.drop_duplicates(subset=['route_id'], keep='first', inplace=True)

# Rule R2: Fill missing route_name with placeholder
null_name = routes['route_name'].isnull().sum() if 'route_name' in routes.columns else 0
if null_name > 0:
    routes['route_name'].fillna('UNKNOWN_ROUTE', inplace=True)
    log_issue('routes', 'route_name', 'missing_value', null_name, 'filled UNKNOWN_ROUTE')

print('Shape after cleaning:', routes.shape)
routes.to_csv(PROCESSED_DIR + 'routes_clean.csv', index=False)
print('Saved: routes_clean.csv')

---
## 3. stops.csv

In [ ]:
# ─── Cell 8: Load & inspect stops.csv ──────────────────────────────────────
stops_raw = pd.read_csv(RAW_DIR + 'stops.csv')
print('Shape (raw):', stops_raw.shape)
print('Nulls:\n', stops_raw.isnull().sum())
stops_raw.head(3)

In [ ]:
# ─── Cell 9: Clean stops ────────────────────────────────────────────────────
stops = stops_raw.copy()

# Rule S1: Drop duplicate stop_id
dup_s = stops.duplicated(subset=['stop_id']).sum()
if dup_s > 0:
    log_issue('stops', 'stop_id', 'duplicate_pk', dup_s, 'dropped duplicates')
stops.drop_duplicates(subset=['stop_id'], keep='first', inplace=True)

# Rule S2: Validate latitude/longitude ranges
if 'latitude' in stops.columns and 'longitude' in stops.columns:
    invalid_lat = stops[(stops['latitude'] < -90) | (stops['latitude'] > 90)].shape[0]
    invalid_lon = stops[(stops['longitude'] < -180) | (stops['longitude'] > 180)].shape[0]
    if invalid_lat > 0:
        stops.loc[(stops['latitude'] < -90) | (stops['latitude'] > 90), 'latitude'] = np.nan
        log_issue('stops', 'latitude', 'out_of_range', invalid_lat, 'set to NaN')
    if invalid_lon > 0:
        stops.loc[(stops['longitude'] < -180) | (stops['longitude'] > 180), 'longitude'] = np.nan
        log_issue('stops', 'longitude', 'out_of_range', invalid_lon, 'set to NaN')

print('Shape after cleaning:', stops.shape)
stops.to_csv(PROCESSED_DIR + 'stops_clean.csv', index=False)
print('Saved: stops_clean.csv')

---
## 4. vehicles.csv

In [ ]:
# ─── Cell 10: Load & inspect vehicles.csv ───────────────────────────────────
vehicles_raw = pd.read_csv(RAW_DIR + 'vehicles.csv')
print('Shape (raw):', vehicles_raw.shape)
print('Nulls:\n', vehicles_raw.isnull().sum())
vehicles_raw.head(3)

In [ ]:
# ─── Cell 11: Clean vehicles ────────────────────────────────────────────────
vehicles = vehicles_raw.copy()

# Rule V1: Drop duplicate vehicle_id
dup_v = vehicles.duplicated(subset=['vehicle_id']).sum()
if dup_v > 0:
    log_issue('vehicles', 'vehicle_id', 'duplicate_pk', dup_v, 'dropped duplicates')
vehicles.drop_duplicates(subset=['vehicle_id'], keep='first', inplace=True)

# Rule V2: Capacity must be positive
if 'capacity' in vehicles.columns:
    invalid_cap = vehicles[vehicles['capacity'] <= 0].shape[0]
    if invalid_cap > 0:
        median_cap = vehicles[vehicles['capacity'] > 0]['capacity'].median()
        vehicles.loc[vehicles['capacity'] <= 0, 'capacity'] = median_cap
        log_issue('vehicles', 'capacity', 'non_positive_value', invalid_cap,
                  f'replaced with median = {median_cap}')

print('Shape after cleaning:', vehicles.shape)
vehicles.to_csv(PROCESSED_DIR + 'vehicles_clean.csv', index=False)
print('Saved: vehicles_clean.csv')

---
## 5. route_stops.csv

In [ ]:
# ─── Cell 12: Load & inspect route_stops.csv ────────────────────────────────
route_stops_raw = pd.read_csv(RAW_DIR + 'route_stops.csv')
print('Shape (raw):', route_stops_raw.shape)
print('Nulls:\n', route_stops_raw.isnull().sum())
route_stops_raw.head(3)

In [ ]:
# ─── Cell 13: Clean route_stops ─────────────────────────────────────────────
route_stops = route_stops_raw.copy()

# Rule RS1: Drop rows with null route_id or stop_id (FK integrity)
null_fk = route_stops[route_stops['route_id'].isnull() | route_stops['stop_id'].isnull()].shape[0]
if null_fk > 0:
    log_issue('route_stops', 'route_id/stop_id', 'null_foreign_key', null_fk, 'dropped rows')
route_stops.dropna(subset=['route_id', 'stop_id'], inplace=True)

# Rule RS2: Validate route_id against known routes
valid_route_ids = set(routes['route_id'].values)
invalid_route_ref = route_stops[~route_stops['route_id'].isin(valid_route_ids)].shape[0]
if invalid_route_ref > 0:
    log_issue('route_stops', 'route_id', 'invalid_foreign_key', invalid_route_ref,
              'flagged as INVALID_ROUTE_REF')
    route_stops.loc[~route_stops['route_id'].isin(valid_route_ids), 'route_id'] = 'INVALID_ROUTE_REF'

# Rule RS3: Drop duplicate (route_id, stop_id, stop_sequence) combos
dup_rs = route_stops.duplicated().sum()
if dup_rs > 0:
    log_issue('route_stops', 'all_cols', 'duplicate_row', dup_rs, 'dropped duplicates')
route_stops.drop_duplicates(inplace=True)

print('Shape after cleaning:', route_stops.shape)
route_stops.to_csv(PROCESSED_DIR + 'route_stops_clean.csv', index=False)
print('Saved: route_stops_clean.csv')

---
## 6. schedules.csv

In [ ]:
# ─── Cell 14: Load & inspect schedules.csv ──────────────────────────────────
schedules_raw = pd.read_csv(RAW_DIR + 'schedules.csv')
print('Shape (raw):', schedules_raw.shape)
print('Nulls:\n', schedules_raw.isnull().sum())
schedules_raw.head(3)

In [ ]:
# ─── Cell 15: Clean schedules ────────────────────────────────────────────────
schedules = schedules_raw.copy()

# Rule SC1: Parse scheduled_departure and scheduled_arrival as datetime
for col in ['scheduled_departure', 'scheduled_arrival']:
    if col in schedules.columns:
        schedules[col] = pd.to_datetime(schedules[col], errors='coerce')
        bad = schedules[col].isnull().sum()
        if bad > 0:
            log_issue('schedules', col, 'invalid_datetime', bad, 'set to NaT')

# Rule SC2: Departure must be before arrival (if both columns exist)
if 'scheduled_departure' in schedules.columns and 'scheduled_arrival' in schedules.columns:
    dep_after_arr = schedules[
        schedules['scheduled_departure'] >= schedules['scheduled_arrival']
    ].shape[0]
    if dep_after_arr > 0:
        log_issue('schedules', 'scheduled_departure', 'departure_after_arrival',
                  dep_after_arr, 'flagged with is_invalid_schedule=True')
        schedules['is_invalid_schedule'] = (
            schedules['scheduled_departure'] >= schedules['scheduled_arrival']
        )

print('Shape after cleaning:', schedules.shape)
schedules.to_csv(PROCESSED_DIR + 'schedules_clean.csv', index=False)
print('Saved: schedules_clean.csv')

---
## 7. service_calendar.csv

In [ ]:
# ─── Cell 16: Load & inspect service_calendar.csv ──────────────────────────
service_calendar_raw = pd.read_csv(RAW_DIR + 'service_calendar.csv')
print('Shape (raw):', service_calendar_raw.shape)
print('Nulls:\n', service_calendar_raw.isnull().sum())
service_calendar_raw

In [ ]:
# ─── Cell 17: Clean service_calendar ────────────────────────────────────────
service_calendar = service_calendar_raw.copy()

# Rule CAL1: Drop fully duplicate rows
dup_cal = service_calendar.duplicated().sum()
if dup_cal > 0:
    log_issue('service_calendar', 'all_cols', 'duplicate_row', dup_cal, 'dropped duplicates')
service_calendar.drop_duplicates(inplace=True)

# Rule CAL2: Parse start_date / end_date
for col in ['start_date', 'end_date']:
    if col in service_calendar.columns:
        service_calendar[col] = pd.to_datetime(service_calendar[col], errors='coerce')

print('Shape after cleaning:', service_calendar.shape)
service_calendar.to_csv(PROCESSED_DIR + 'service_calendar_clean.csv', index=False)
print('Saved: service_calendar_clean.csv')

---
## 8. trips.csv

In [ ]:
# ─── Cell 18: Load trips.csv (chunked for large file) ───────────────────────
print('Loading trips.csv ...')
trips_raw = pd.read_csv(RAW_DIR + 'trips.csv')
print('Shape (raw):', trips_raw.shape)
print('Nulls:\n', trips_raw.isnull().sum())
trips_raw.head(3)

In [ ]:
# ─── Cell 19: Clean trips ────────────────────────────────────────────────────
trips = trips_raw.copy()

# Rule T1: Drop duplicate trip_id
dup_t = trips.duplicated(subset=['trip_id']).sum()
if dup_t > 0:
    log_issue('trips', 'trip_id', 'duplicate_pk', dup_t, 'dropped duplicates')
trips.drop_duplicates(subset=['trip_id'], keep='first', inplace=True)

# Rule T2: Validate route_id FK
valid_route_ids = set(routes['route_id'].values)
invalid_route_ref_t = trips[~trips['route_id'].isin(valid_route_ids)].shape[0]
if invalid_route_ref_t > 0:
    log_issue('trips', 'route_id', 'invalid_foreign_key', invalid_route_ref_t, 'flagged')
    trips['route_id_valid'] = trips['route_id'].isin(valid_route_ids)

# Rule T3: Parse actual_departure and actual_arrival as datetime
for col in ['actual_departure', 'actual_arrival', 'scheduled_departure', 'scheduled_arrival']:
    if col in trips.columns:
        trips[col] = pd.to_datetime(trips[col], errors='coerce')
        bad = trips[col].isnull().sum()
        if bad > 0:
            log_issue('trips', col, 'invalid_datetime', bad, 'set to NaT')

# Rule T4: Departure before arrival check
if 'actual_departure' in trips.columns and 'actual_arrival' in trips.columns:
    bad_order = trips[
        trips['actual_departure'] >= trips['actual_arrival']
    ].shape[0]
    if bad_order > 0:
        trips['is_invalid_trip_time'] = trips['actual_departure'] >= trips['actual_arrival']
        log_issue('trips', 'actual_departure/actual_arrival', 'departure_after_arrival',
                  bad_order, 'flagged with is_invalid_trip_time=True')

print('Shape after cleaning:', trips.shape)
trips.to_csv(PROCESSED_DIR + 'trips_clean.csv', index=False)
print('Saved: trips_clean.csv')

---
## 9. delays.csv

In [ ]:
# ─── Cell 20: Load & inspect delays.csv ─────────────────────────────────────
print('Loading delays.csv ...')
delays_raw = pd.read_csv(RAW_DIR + 'delays.csv')
print('Shape (raw):', delays_raw.shape)
print('Nulls:\n', delays_raw.isnull().sum())
delays_raw.head(3)

In [ ]:
# ─── Cell 21: Clean delays ───────────────────────────────────────────────────
delays = delays_raw.copy()

# Rule D1: Drop duplicate delay_id
dup_d = delays.duplicated(subset=['delay_id']).sum()
if dup_d > 0:
    log_issue('delays', 'delay_id', 'duplicate_pk', dup_d, 'dropped duplicates')
delays.drop_duplicates(subset=['delay_id'], keep='first', inplace=True)

# Rule D2: delay_minutes must be >= 0; negative values flagged
if 'delay_minutes' in delays.columns:
    neg_delay = delays[delays['delay_minutes'] < 0].shape[0]
    if neg_delay > 0:
        delays.loc[delays['delay_minutes'] < 0, 'delay_minutes'] = 0
        log_issue('delays', 'delay_minutes', 'negative_value', neg_delay,
                  'clamped to 0 (early arrivals treated as zero delay)')

# Rule D3: Outlier cap — delays > 300 min capped (likely data entry error)
if 'delay_minutes' in delays.columns:
    extreme_delay = delays[delays['delay_minutes'] > 300].shape[0]
    if extreme_delay > 0:
        delays.loc[delays['delay_minutes'] > 300, 'delay_minutes'] = 300
        log_issue('delays', 'delay_minutes', 'extreme_outlier', extreme_delay,
                  'capped at 300 minutes')

print('Shape after cleaning:', delays.shape)
delays.to_csv(PROCESSED_DIR + 'delays_clean.csv', index=False)
print('Saved: delays_clean.csv')

---
## 10. tickets.csv

In [ ]:
# ─── Cell 22: Load tickets.csv (large file — ~125 MB) ───────────────────────
print('Loading tickets.csv ... (this may take a moment)')
tickets_raw = pd.read_csv(RAW_DIR + 'tickets.csv')
print('Shape (raw):', tickets_raw.shape)
print('Nulls:\n', tickets_raw.isnull().sum())
tickets_raw.head(3)

In [ ]:
# ─── Cell 23: Clean tickets ──────────────────────────────────────────────────
tickets = tickets_raw.copy()

# Rule TK1: Drop duplicate ticket_id
dup_tk = tickets.duplicated(subset=['ticket_id']).sum()
if dup_tk > 0:
    log_issue('tickets', 'ticket_id', 'duplicate_pk', dup_tk, 'dropped duplicates')
tickets.drop_duplicates(subset=['ticket_id'], keep='first', inplace=True)

# Rule TK2: Validate passenger_id FK
valid_pax_ids = set(passengers['passenger_id'].values)
invalid_pax = tickets[~tickets['passenger_id'].isin(valid_pax_ids)].shape[0]
if invalid_pax > 0:
    log_issue('tickets', 'passenger_id', 'invalid_foreign_key', invalid_pax, 'flagged')
    tickets['pax_id_valid'] = tickets['passenger_id'].isin(valid_pax_ids)

# Rule TK3: Parse purchase_time as datetime
if 'purchase_time' in tickets.columns:
    tickets['purchase_time'] = pd.to_datetime(tickets['purchase_time'], errors='coerce')
    bad_ts = tickets['purchase_time'].isnull().sum()
    if bad_ts > 0:
        log_issue('tickets', 'purchase_time', 'invalid_datetime', bad_ts, 'set to NaT')

# Rule TK4: fare_amount must be >= 0
if 'fare_amount' in tickets.columns:
    neg_fare = tickets[tickets['fare_amount'] < 0].shape[0]
    if neg_fare > 0:
        tickets.loc[tickets['fare_amount'] < 0, 'fare_amount'] = np.nan
        log_issue('tickets', 'fare_amount', 'negative_fare', neg_fare, 'set to NaN')

print('Shape after cleaning:', tickets.shape)
tickets.to_csv(PROCESSED_DIR + 'tickets_clean.csv', index=False)
print('Saved: tickets_clean.csv')

---
## 11. passenger_counts.csv

In [ ]:
# ─── Cell 24: Load passenger_counts.csv (~198 MB — largest file) ────────────
print('Loading passenger_counts.csv ... (largest file, please wait)')
pax_counts_raw = pd.read_csv(RAW_DIR + 'passenger_counts.csv')
print('Shape (raw):', pax_counts_raw.shape)
print('Nulls:\n', pax_counts_raw.isnull().sum())
pax_counts_raw.head(3)

In [ ]:
# ─── Cell 25: Clean passenger_counts ────────────────────────────────────────
pax_counts = pax_counts_raw.copy()

# Rule PC1: boarding_count and alighting_count must be >= 0
for col in ['boarding_count', 'alighting_count']:
    if col in pax_counts.columns:
        neg = pax_counts[pax_counts[col] < 0].shape[0]
        if neg > 0:
            pax_counts.loc[pax_counts[col] < 0, col] = 0
            log_issue('passenger_counts', col, 'negative_count', neg, 'clamped to 0')

# Rule PC2: occupancy_pct must be 0–200 (max 200% = overcrowded)
if 'occupancy_pct' in pax_counts.columns:
    bad_occ = pax_counts[pax_counts['occupancy_pct'] > 200].shape[0]
    if bad_occ > 0:
        pax_counts.loc[pax_counts['occupancy_pct'] > 200, 'occupancy_pct'] = 200
        log_issue('passenger_counts', 'occupancy_pct', 'extreme_outlier', bad_occ,
                  'capped at 200%')

# Rule PC3: Drop duplicate rows
dup_pc = pax_counts.duplicated().sum()
if dup_pc > 0:
    log_issue('passenger_counts', 'all_cols', 'duplicate_row', dup_pc, 'dropped')
pax_counts.drop_duplicates(inplace=True)

print('Shape after cleaning:', pax_counts.shape)
pax_counts.to_csv(PROCESSED_DIR + 'passenger_counts_clean.csv', index=False)
print('Saved: passenger_counts_clean.csv')

---
## 12. gps_events.csv

In [ ]:
# ─── Cell 26: Load gps_events.csv (~64 MB) ──────────────────────────────────
print('Loading gps_events.csv ...')
gps_raw = pd.read_csv(RAW_DIR + 'gps_events.csv')
print('Shape (raw):', gps_raw.shape)
print('Nulls:\n', gps_raw.isnull().sum())
gps_raw.head(3)

In [ ]:
# ─── Cell 27: Clean gps_events ──────────────────────────────────────────────
gps = gps_raw.copy()

# Rule G1: Validate latitude/longitude
if 'latitude' in gps.columns and 'longitude' in gps.columns:
    invalid_lat = gps[(gps['latitude'] < -90) | (gps['latitude'] > 90)].shape[0]
    invalid_lon = gps[(gps['longitude'] < -180) | (gps['longitude'] > 180)].shape[0]
    if invalid_lat > 0:
        gps.loc[(gps['latitude'] < -90) | (gps['latitude'] > 90), 'latitude'] = np.nan
        log_issue('gps_events', 'latitude', 'out_of_range', invalid_lat, 'set to NaN')
    if invalid_lon > 0:
        gps.loc[(gps['longitude'] < -180) | (gps['longitude'] > 180), 'longitude'] = np.nan
        log_issue('gps_events', 'longitude', 'out_of_range', invalid_lon, 'set to NaN')

# Rule G2: Parse timestamp
if 'timestamp' in gps.columns:
    gps['timestamp'] = pd.to_datetime(gps['timestamp'], errors='coerce')
    bad_ts = gps['timestamp'].isnull().sum()
    if bad_ts > 0:
        log_issue('gps_events', 'timestamp', 'invalid_datetime', bad_ts, 'set to NaT')

# Rule G3: speed_kmh must be 0–200
if 'speed_kmh' in gps.columns:
    neg_spd = gps[gps['speed_kmh'] < 0].shape[0]
    high_spd = gps[gps['speed_kmh'] > 200].shape[0]
    if neg_spd > 0:
        gps.loc[gps['speed_kmh'] < 0, 'speed_kmh'] = 0
        log_issue('gps_events', 'speed_kmh', 'negative_speed', neg_spd, 'clamped to 0')
    if high_spd > 0:
        gps.loc[gps['speed_kmh'] > 200, 'speed_kmh'] = np.nan
        log_issue('gps_events', 'speed_kmh', 'extreme_speed', high_spd, 'set to NaN')

# Rule G4: Drop duplicate rows
dup_gps = gps.duplicated().sum()
if dup_gps > 0:
    log_issue('gps_events', 'all_cols', 'duplicate_row', dup_gps, 'dropped')
gps.drop_duplicates(inplace=True)

print('Shape after cleaning:', gps.shape)
gps.to_csv(PROCESSED_DIR + 'gps_events_clean.csv', index=False)
print('Saved: gps_events_clean.csv')

---
## Final: Cleaning Audit Report

In [ ]:
# ─── Cell 28: Print & Save Cleaning Audit Log ────────────────────────────────
audit_df = pd.DataFrame(cleaning_log)

print('='*70)
print('         URBANTRANSIT IQ — DATA CLEANING AUDIT REPORT')
print('='*70)

if audit_df.empty:
    print('No issues detected! All tables passed quality checks.')
else:
    print(f'Total issues detected & resolved: {len(audit_df)}')
    print()
    print(audit_df.to_string(index=False))

# Save audit log
audit_df.to_csv(PROCESSED_DIR + 'cleaning_audit_log.csv', index=False)
print('\n✅ Audit log saved to processed_data/cleaning_audit_log.csv')

In [ ]:
# ─── Cell 29: Summary — Processed Files List ────────────────────────────────
import os
processed_files = [f for f in os.listdir(PROCESSED_DIR) if f.endswith('.csv')]
print('Files saved in processed_data/:')
for f in sorted(processed_files):
    size_kb = os.path.getsize(PROCESSED_DIR + f) // 1024
    print(f'  {f:40s}  {size_kb:>8} KB')